# A- YouTube Data Collection
This part collects video metadata, top-level comments and replies from the YouTube Data API v3
and saves them as raw JSONL files, which are later uploaded to HDFS and processed with Spark.

## 1. Setup and Configuration
Imports, API key placeholder, output folder and a quota counter (daily limit: 10,000 units)

In [ ]:
import os, json
from datetime import datetime, timedelta, timezone
import requests

API_KEY = "your api key"
BASE_URL = "https://www.googleapis.com/youtube/v3"

OUT_DIR = "raw"
os.makedirs(OUT_DIR, exist_ok=True)
COLLECTED_AT = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
units_used = 0   # quota counter (daily limit = 10,000 units)

## 2. Defining the API Request and Save Functions
call_api sends one request and tracks quota usage (search = 100 units, other endpoints = 1 unit)
save appends results to a JSONL file and adds a collection timestamp.

In [2]:
#Sends one request to the YouTube API and returns the JSON answer

def call_api(service, params):
    global units_used
    units_used += 100 if service == "search" else 1
    params["key"] = API_KEY
    response = requests.get(f"{BASE_URL}/{service}", params=params, timeout=30)
    if response.status_code != 200:
        try:
            reason = response.json()["error"]["errors"][0]["reason"]
        except Exception:
            reason = response.text[:100]
        print(f"  {service} error {response.status_code}: {reason}")
        return None
    return response.json()

#Appends a list of dicts to a .jsonl file (one JSON object per line)


def save(file_name, rows):
    with open(f"{OUT_DIR}/{file_name}.jsonl", "a", encoding="utf-8") as f:
        for row in rows:
            row["collected_at"] = COLLECTED_AT
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

## 3. Searching for Videos
Finds videos aged 30–90 days in a given category using the search endpoint.

In [3]:
def get_video_ids(category_id, region="IE", max_results=50,
                  min_age_days=30, max_age_days=90):
    now = datetime.now(timezone.utc)
    params = {
        "part": "id",
        "type": "video",
        "videoCategoryId": category_id,
        "regionCode": region,
        "relevanceLanguage": "en",
        "order": "relevance",
        "publishedAfter":  (now - timedelta(days=max_age_days)).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "publishedBefore": (now - timedelta(days=min_age_days)).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "maxResults": max_results,
    }
    data = call_api("search", params)
    if data is None:
        return []
    return [item["id"]["videoId"] for item in data["items"]]

## 4. Collecting Video Metadata
Retrieves title, description, tags, category and publish date using the videos endpoint.

In [4]:
def get_videos(video_ids):
    rows = []
    for i in range(0, len(video_ids), 50):          # max 50 ids per request
        data = call_api("videos", {"part": "snippet", "id": ",".join(video_ids[i:i+50])})
        if data is None:
            continue
        for item in data["items"]:
            s = item["snippet"]
            rows.append({
                "video_id": item["id"],
                "video_publish_at": s["publishedAt"],
                "video_category": s.get("categoryId"),
                "video_title": s.get("title"),
                "video_description": s.get("description"),
                "video_tags": s.get("tags", []),     # some videos have no tags
            })
    save("videos", rows)
    return rows

## 5. Collecting Top-Level Comments
Retrieves comments and their reply counts using the commentThreads endpoint.

In [5]:
def get_comments(video_id, max_pages=5):
    rows, page_token = [], None
    for _ in range(max_pages):
        params = {"part": "snippet", "videoId": video_id, "maxResults": 100,
                  "textFormat": "plainText"}
        if page_token:
            params["pageToken"] = page_token
        data = call_api("commentThreads", params)
        if data is None:                             # e.g. comments disabled
            break
        for item in data["items"]:
            top = item["snippet"]["topLevelComment"]["snippet"]
            rows.append({
                "comment_id": item["id"],
                "video_id": video_id,
                "comment_text": top["textDisplay"],
                "comment_publish_at": top["publishedAt"],
                "total_reply_count": item["snippet"]["totalReplyCount"],
            })
        page_token = data.get("nextPageToken")
        if not page_token:
            break
    save("comments", rows)
    return rows

## 6. Collecting Replies
Retrieves the replies to each comment using the comments endpoint.

In [6]:
def get_reply_dates(comment_id, max_pages=3):
    rows, page_token = [], None
    for _ in range(max_pages):
        params = {"part": "snippet", "parentId": comment_id, "maxResults": 100}
        if page_token:
            params["pageToken"] = page_token
        data = call_api("comments", params)
        if data is None:
            break
        for item in data["items"]:
            rows.append({
                "reply_id": item["id"],
                "parent_id": comment_id,
                "reply_publish_at": item["snippet"]["publishedAt"],
            })
        page_token = data.get("nextPageToken")
        if not page_token:
            break
    save("replies", rows)
    return rows

## 7. Running the Collection
Loops over the selected categories, collects all data and reports the quota used.

The collection was run on three days because of the daily API quota (10,000 units), with different settings:
- Day 1: categories 10, 17, 20, 24, 28, 'order' = "viewCount"
- Day 2: categories 1, 22, 26, 'order' = "relevance"
- Day 3: categories 2, 15, 19, 23, 25, 27, 'order' = "relevance", "max_pages" = 5

The code below shows the Day 3 configuration. All runs appended to the same JSONL files.

In [7]:
CATEGORY_IDS = ["2", "15", "19", "23", "25", "27"]
REGION = "IE"
VIDEOS_PER_CATEGORY = 50  

for category_id in CATEGORY_IDS:
    video_ids = get_video_ids(category_id, REGION, VIDEOS_PER_CATEGORY)
    videos = get_videos(video_ids)
    print(f"Category {category_id}: {len(videos)} videos")

    for video in videos:
        comments = get_comments(video["video_id"])

        n_replies = 0
        for c in comments:
            if c["total_reply_count"] > 0:
                n_replies += len(get_reply_dates(c["comment_id"]))
        print(f"  {video['video_id']}: {len(comments)} comments, {n_replies} replies | units used: {units_used}")

print("Total quota units used: ", units_used)

Category 2: 50 videos
  7HJ7a9vQM2U: 500 comments, 190 replies | units used: 115
  JGbKWChMGuM: 176 comments, 142 replies | units used: 196
  eos_8XSLLyI: 500 comments, 5 replies | units used: 206
  KhE1lrlvHKA: 82 comments, 22 replies | units used: 227
  9Xy4QgPbAcE: 22 comments, 9 replies | units used: 236
  DTnLhFSaeLw: 500 comments, 8 replies | units used: 247
  dT0QLC7Cv8s: 94 comments, 57 replies | units used: 294
  5ihepBmd0Jw: 500 comments, 9 replies | units used: 306
  WBRh4gfnPbc: 499 comments, 14 replies | units used: 324
  gUz5fmKNV7c: 0 comments, 0 replies | units used: 325
  c6zwwlYII88: 500 comments, 0 replies | units used: 330
  0Vf_ngk2Spw: 16 comments, 5 replies | units used: 336
  vSKnONk-nbQ: 500 comments, 76 replies | units used: 349
  IOGNZKq2rM8: 388 comments, 72 replies | units used: 368
  H1s5SgfkqA8: 500 comments, 1 replies | units used: 374
  TLAVHTJHCCo: 500 comments, 11 replies | units used: 388
  6tnxX9_Vhys: 500 comments, 126 replies | units used: 413
  u

  7rd6HWqvaHQ: 500 comments, 28 replies | units used: 3220
  9Zdudi3yNK0: 500 comments, 7 replies | units used: 3230
  XKnmhvNpIvw: 499 comments, 24 replies | units used: 3256
  mlJDEtoFeNo: 500 comments, 19 replies | units used: 3277
  stM4bbBGxEM: 499 comments, 3 replies | units used: 3285
  Pv7i0jzQ4UY: 500 comments, 9 replies | units used: 3293
  3qGHR8ZZd9w: 500 comments, 7 replies | units used: 3304
  yGw0yrpOFL0: 500 comments, 3 replies | units used: 3311
  lfLNJzk0XFE: 499 comments, 167 replies | units used: 3367
  umzHnGqZXus: 500 comments, 7 replies | units used: 3377
  HJY2LWJbGjI: 500 comments, 16 replies | units used: 3397
Category 23: 50 videos
  dxUnsK8mw20: 500 comments, 9 replies | units used: 3508
  commentThreads error 403: commentsDisabled
  IBzk_Zv35IU: 0 comments, 0 replies | units used: 3509
  dez812Q7QP8: 500 comments, 7 replies | units used: 3517
  v_gML4r1NeQ: 477 comments, 55 replies | units used: 3539
  Q4_6ngsvtRM: 500 comments, 4 replies | units used: 3548

  dRou8TJouyU: 500 comments, 44 replies | units used: 5691
  D8SbSJwIsgQ: 500 comments, 17 replies | units used: 5710
  Ru__9nZmqlE: 500 comments, 190 replies | units used: 5862
  BhiDh6tzs84: 500 comments, 11 replies | units used: 5876
  2IPhnZkEpmA: 500 comments, 2 replies | units used: 5883
  M42Ub81jptM: 500 comments, 161 replies | units used: 5893
  69d59OApvAY: 500 comments, 13 replies | units used: 5908
  HX29TphGKao: 410 comments, 78 replies | units used: 5932
  VikN-biebvg: 500 comments, 89 replies | units used: 5967
  k6YwbDeBYlQ: 4 comments, 1 replies | units used: 5969
  kNsKk80oIYU: 148 comments, 33 replies | units used: 5988
  commentThreads error 403: commentsDisabled
  bqle6Cq7fR0: 0 comments, 0 replies | units used: 5989
  OmodSjkQUEg: 500 comments, 2 replies | units used: 5996
  tIjyxRVi3W4: 500 comments, 3 replies | units used: 6004
  _B7pwoc-MC4: 500 comments, 12 replies | units used: 6021
  commentThreads error 403: commentsDisabled
  LxtyaBsL4Lk: 0 comments, 0 rep

## 8. Verifying the Collected Raw Data
Reads the three JSONL files back with pandas to check that they were saved correctly and to count the records.

In [5]:
import pandas as pd
for f in ["videos", "comments", "replies"]:
    df = pd.read_json(f"raw/{f}.jsonl", lines=True)
    print(f, df.shape)

videos (703, 7)
comments (168567, 6)
replies (30982, 4)


## 9. Uploading to HDFS and Verifying with Spark

The raw files were uploaded to HDFS from the terminal:

    hdfs dfs -mkdir -p /ca1/raw

    hdfs dfs -put raw/*.jsonl /ca1/raw/

Spark then reads them back from HDFS to confirm the upload and inspect the schema.

In [6]:
# Connect to the local Spark installation

import os, sys, glob
os.environ["SPARK_HOME"] = "/usr/local/spark"
os.environ["PYSPARK_PYTHON"] = sys.executable
sys.path.insert(0, "/usr/local/spark/python")
sys.path.insert(0, glob.glob("/usr/local/spark/python/lib/py4j-*.zip")[0])

# Read the raw JSONL files from HDFS


from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("CA1_YouTube").getOrCreate()

HDFS = "hdfs://127.0.0.1:9000/ca1/raw"
videos   = spark.read.json(f"{HDFS}/videos.jsonl")
comments = spark.read.json(f"{HDFS}/comments.jsonl")
replies  = spark.read.json(f"{HDFS}/replies.jsonl")

print("videos:", videos.count())
print("comments:", comments.count())
print("replies:", replies.count())
comments.printSchema()

videos: 703
comments: 168567
replies: 30982
root
 |-- collected_at: string (nullable = true)
 |-- comment_id: string (nullable = true)
 |-- comment_publish_at: string (nullable = true)
 |-- comment_text: string (nullable = true)
 |-- total_reply_count: long (nullable = true)
 |-- video_id: string (nullable = true)



# B- Big Data Processing with Apache Spark

This part reads the raw YouTube data from HDFS, cleans and joins it, creates features and the target label, and saves the final dataset to HDFS as partitioned Parquet files.

## 1. Removing Duplicate Records

In [7]:
print("Before -> videos:", videos.count(), "| comments:", comments.count(), "| replies:", replies.count())

videos   = videos.dropDuplicates(["video_id"])
comments = comments.dropDuplicates(["comment_id"])
replies  = replies.dropDuplicates(["reply_id"])

print("After  -> videos:", videos.count(), "| comments:", comments.count(), "| replies:", replies.count())

Before -> videos: 703 | comments: 168567 | replies: 30982
After  -> videos: 700 | comments: 168152 | replies: 30232


## 2. Converting Strings to Timestamps

In [8]:
from pyspark.sql import functions as F

videos   = videos.withColumn("video_publish_at", F.to_timestamp("video_publish_at"))
comments = comments.withColumn("comment_publish_at", F.to_timestamp("comment_publish_at")) \
                   .withColumn("collected_at", F.to_timestamp("collected_at"))
replies  = replies.withColumn("reply_publish_at", F.to_timestamp("reply_publish_at"))

videos.select("video_publish_at").printSchema()
comments.select("comment_publish_at", "collected_at").printSchema()
replies.select("reply_publish_at").printSchema()

root
 |-- video_publish_at: timestamp (nullable = true)

root
 |-- comment_publish_at: timestamp (nullable = true)
 |-- collected_at: timestamp (nullable = true)

root
 |-- reply_publish_at: timestamp (nullable = true)



## 3. Checking for Missing Values

In [9]:
def count_nulls(df, name):
    print(f"--- {name} ---")
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

count_nulls(videos, "videos")
count_nulls(comments, "comments")
count_nulls(replies, "replies")

--- videos ---
+------------+--------------+-----------------+--------+----------------+----------+-----------+
|collected_at|video_category|video_description|video_id|video_publish_at|video_tags|video_title|
+------------+--------------+-----------------+--------+----------------+----------+-----------+
|           0|             0|                0|       0|               0|         0|          0|
+------------+--------------+-----------------+--------+----------------+----------+-----------+

--- comments ---


+------------+----------+------------------+------------+-----------------+--------+
|collected_at|comment_id|comment_publish_at|comment_text|total_reply_count|video_id|
+------------+----------+------------------+------------+-----------------+--------+
|           0|         0|                 0|           0|                0|       0|
+------------+----------+------------------+------------+-----------------+--------+

--- replies ---
+------------+---------+--------+----------------+
|collected_at|parent_id|reply_id|reply_publish_at|
+------------+---------+--------+----------------+
|           0|        0|       0|               0|
+------------+---------+--------+----------------+



## 4. Checking for Empty Strings and Empty Arrays

In [10]:
print("Empty video_description:", videos.filter(F.trim("video_description") == "").count())
print("Empty video_tags:       ", videos.filter(F.size("video_tags") == 0).count())
print("Empty video_title:      ", videos.filter(F.trim("video_title") == "").count())
print("Empty comment_text:     ", comments.filter(F.trim("comment_text") == "").count())

Empty video_description: 237
Empty video_tags:        425
Empty video_title:       0
Empty comment_text:      3


## 5. Removing Empty Comments

In [11]:
print("Before:", comments.count())
comments = comments.filter(F.trim("comment_text") != "")
print("After: ", comments.count())

Before: 168152
After:  168149


## 6. Joining Comments with Video Metadata

In [12]:
video_cols = videos.select("video_id", "video_category", "video_title",
                           "video_description", "video_tags", "video_publish_at")

df = comments.join(video_cols, on="video_id", how="inner")

print("Comments:", comments.count())
print("After join:", df.count())
df.printSchema()

Comments: 168149
After join: 168149
root
 |-- video_id: string (nullable = true)
 |-- collected_at: timestamp (nullable = true)
 |-- comment_id: string (nullable = true)
 |-- comment_publish_at: timestamp (nullable = true)
 |-- comment_text: string (nullable = true)
 |-- total_reply_count: long (nullable = true)
 |-- video_category: string (nullable = true)
 |-- video_title: string (nullable = true)
 |-- video_description: string (nullable = true)
 |-- video_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- video_publish_at: timestamp (nullable = true)



## 7. Feature Engineering

Three new input features are created from the timestamps:

1-video_age_at_comment : hours between the video's publication and the comment. Hours are used because most comments arrive within the first few days, so days would be too coarse.

2-comment_hour : hour of the day the comment was posted (0–23, UTC).

3-comment_weekday: day of the week (1 = Sunday, 7 = Saturday).

In [16]:
df = df.withColumn("video_age_at_comment",
                   (F.col("comment_publish_at").cast("long") - F.col("video_publish_at").cast("long")) / 3600) \
       .withColumn("comment_hour", F.hour("comment_publish_at")) \
       .withColumn("comment_weekday", F.dayofweek("comment_publish_at"))

df.select("video_age_at_comment", "comment_hour", "comment_weekday").describe().show()

+-------+--------------------+------------------+------------------+
|summary|video_age_at_comment|      comment_hour|   comment_weekday|
+-------+--------------------+------------------+------------------+
|  count|              168149|            168149|            168149|
|   mean|   858.8872835732326|11.582923478581495| 4.130039429315667|
| stddev|   534.5889405836505|  6.63888524083529|1.9950287877424133|
|    min| -0.6316666666666667|                 0|                 1|
|    max|  2158.7361111111113|                23|                 7|
+-------+--------------------+------------------+------------------+



## 8. Handling Negative Video Age
video_age_at_comment should not be negative. These 9 rows are removed.

In [17]:
print("Negative video_age_at_comment:", df.filter(F.col("video_age_at_comment") < 0).count())

df = df.filter(F.col("video_age_at_comment") >= 0)
print("After:", df.count())

Negative video_age_at_comment: 9
After: 168140


## 9. Extracting the First Reply

Replies are grouped by 'parent_id' and the earliest reply time is kept as 'first_reply_at' for each comment. A left join keeps comments without replies (their value is null).

In [19]:
first_reply = replies.groupBy("parent_id") \
                     .agg(F.min("reply_publish_at").alias("first_reply_at")) \
                     .withColumnRenamed("parent_id", "comment_id")

df = df.join(first_reply, on="comment_id", how="left")

print("Rows:", df.count())
print("Comments with at least one reply:", df.filter(F.col("first_reply_at").isNotNull()).count())

Rows: 168140
Comments with at least one reply: 7002


## 10. Creating the Target Label and Applying the Censoring Filter
Censoring: comments posted less than 7 days before data collection are removed, because their final label is not yet known.

Label: 1 if the first reply arrived within 7 days of the comment, otherwise 0.

In [20]:
SEVEN_DAYS = 7 * 24 * 3600   # seconds

print("Before censoring:", df.count())
df = df.filter(F.col("collected_at").cast("long") - F.col("comment_publish_at").cast("long") >= SEVEN_DAYS)
print("After censoring: ", df.count())

df = df.withColumn("label",
        F.when(F.col("first_reply_at").isNotNull() &
               (F.col("first_reply_at").cast("long") - F.col("comment_publish_at").cast("long") <= SEVEN_DAYS), 1)
         .otherwise(0))

df.groupBy("label").count().show()

Before censoring: 168140


After censoring:  108524


+-----+------+
|label| count|
+-----+------+
|    1|  5010|
|    0|103514|
+-----+------+



The censoring filter removed 59,616 comments (168,140- 59,616 = 108,524, about 35%), because less than 7 days had passed between their posting and the data collection, so their final label could not yet be known.

The final dataset is highly imbalanced: 5,010 comments (4.6%) received a reply within 7 days (label = 1), while 103,514 (95.4%) did not (label = 0).

## 11. Saving the Final Dataset to HDFS (Parquet)

In [21]:
model_columns = ["comment_id", "video_id",
                 "comment_text",                                                        # Group 1
                 "video_category", "video_title", "video_description", "video_tags",   # Group 2
                 "video_age_at_comment", "comment_hour", "comment_weekday",            # Group 3
                 "label"]

model_dataset = df.select(model_columns)

OUTPUT_PATH = "hdfs://127.0.0.1:9000/ca1/processed/comments_labeled"

model_dataset.write.mode("overwrite") \
             .partitionBy("video_category") \
             .parquet(OUTPUT_PATH)

saved_dataset = spark.read.parquet(OUTPUT_PATH)
print("Saved rows:", saved_dataset.count())
saved_dataset.printSchema()

Saved rows: 108524
root
 |-- comment_id: string (nullable = true)
 |-- video_id: string (nullable = true)
 |-- comment_text: string (nullable = true)
 |-- video_title: string (nullable = true)
 |-- video_description: string (nullable = true)
 |-- video_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- video_age_at_comment: double (nullable = true)
 |-- comment_hour: integer (nullable = true)
 |-- comment_weekday: integer (nullable = true)
 |-- label: integer (nullable = true)
 |-- video_category: integer (nullable = true)



## 12. Previewing the Final Dataset
The first five rows of the saved dataset, showing the Group 1–3 input columns and the target label.

In [22]:
saved_dataset.show(5)

+--------------------+-----------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+---------------+-----+--------------+
|          comment_id|   video_id|        comment_text|         video_title|   video_description|          video_tags|video_age_at_comment|comment_hour|comment_weekday|label|video_category|
+--------------------+-----------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+---------------+-----+--------------+
|Ugw--xqQJI_vVrBJo...|pTENB-nPOLA|You did not even ...|Dave's Hot Chicke...|We brought Dave's...|[sosoadventures, ...|  11.814444444444444|          19|              4|    0|            19|
|Ugw-0fkME2CPQ8f8T...|mlJDEtoFeNo|bang gati  ban  n...|Jangan BERTINGKAH...|                    |                  []|  15.036666666666667|           4|              1|    0|            19|
|Ugw-1s1P6wJMeGCbJ...|7rd6HWqvaHQ|                